This notebook contains the code implementing the following:
1. Band-pass filtering of ICA-cleaned EEG signals (with blink artifacts removed) for the Alpha band.
2. A function to compute the Hilbert transform and extract the asynchronous signal from pairs of ICA-cleaned EEG signals.
3. Downsampling the resulting asynchronous signal and applying an anti-aliasing filter.
4. ffDTF for 4 signals: (EEG_faa_cg, EEG_faa_ch, IBI_cg, IBI_ch).
5. Analyzing group differences: (TD, ASD, SUROG across respective movies).

In [11]:
import os
import sys
import numpy as np
import json

import importlib
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

import src.eeg_alpha_ibi_ffdtf
importlib.reload(src.eeg_alpha_ibi_ffdtf)
from src.eeg_alpha_ibi_ffdtf import EEG_IBI_FFDTF_Pipeline

In [12]:
cleaned_signals_folder = Path(r"C:\Users\matit\Studia\Magisterka\SYNCCIN\hyperscanning-signal-analysis\DATA_film_cleaned")
output_ffDTF_folder = Path(r"C:\Users\matit\Studia\Magisterka\SYNCCIN\hyperscanning-signal-analysis\RESULTS\ffDTF")
target_events = ["Brave", "Peppa", "Incredibles"]

In [13]:
AnalysisPipeline = EEG_IBI_FFDTF_Pipeline(
    cleaned_signals_folder,
    output_ffDTF_folder,
    target_events,
    smoke_test = True,
    smoke_dyads_n = 2,
    left_frontal_eeg_channel  = "F3",
    right_frontal_eeg_channel  = "F4",
    fs_downsampled  = 8,
    n_windows  = 3,
    window_size  = None,
    ar_p = 5,
    plot_global_enabled = False,
    save_global_enabled = True,
    plot_windowed_enabled = False,
    save_windowed_enabled  = True
    )


=== Initialization Complete: SMOKE TEST (first 2 dyads) ===
 [INFO] Target events : ['Brave', 'Peppa', 'Incredibles']
 [INFO] Dyads loaded  : 2/60 (W_000, W_001)
 [OK]   EEG files     : 12/346 ready
 [OK]   IBI files     : 12/346 ready



In [14]:
AnalysisPipeline.run_pipeline()

--- Processing dyad: W_000 | Film: Brave ---
 [OK] Loaded EEG (ch) : W_000_EEG_ch_Brave_cleaned.nc
 [OK] Loaded IBI (ch) : W_000_IBI_ch_Brave.nc
 [OK] Loaded EEG (cg) : W_000_EEG_cg_Brave_cleaned.nc
 [OK] Loaded IBI (cg) : W_000_IBI_cg_Brave.nc
 [OK] Pre-processing complete (Alpha -> FAA -> Downsample -> Crop -> Z-Score)
 [INFO] AIC suggested p=7 for global signal. Forcing fixed p=5.
 [INFO] Computing windowed ffDTF (3 windows)...
 [INFO] Computing global ffDTF...
[SAVED] W_000 | Brave --> C:\Users\matit\Studia\Magisterka\SYNCCIN\hyperscanning-signal-analysis\RESULTS\ffDTF\W_000\W_000_Brave_ffDTF.npz

--- Processing dyad: W_000 | Film: Peppa ---
 [OK] Loaded EEG (ch) : W_000_EEG_ch_Peppa_cleaned.nc
 [OK] Loaded IBI (ch) : W_000_IBI_ch_Peppa.nc
 [OK] Loaded EEG (cg) : W_000_EEG_cg_Peppa_cleaned.nc
 [OK] Loaded IBI (cg) : W_000_IBI_cg_Peppa.nc
 [OK] Pre-processing complete (Alpha -> FAA -> Downsample -> Crop -> Z-Score)
 [INFO] AIC suggested p=7 for global signal. Forcing fixed p=5.
 [IN

In [15]:
dyad = "W_000"
film = "Peppa"
file_path = output_ffDTF_folder / dyad / f"{dyad}_{film}_ffDTF.npz"

with np.load(file_path, allow_pickle=True) as data:
    ff_dtf_g = data["ff_dtf_global"]
    ff_dtf_w = data["ff_dtf_windowed"]
    p_opt_g = data["p_opt_g"]

    meta = json.loads(str(data["meta"]))
    print(f"dyad: {meta['dyad']}")
    print(f"film: {meta['film']}")
    print(f"freq_samp: {meta['fs']} Hz")
    print(f"n_windows: {meta['windowing']['n_windows']}")

dyad: W_000
film: Peppa
freq_samp: 8.0 Hz
n_windows: 3
